In [ ]:
## Notebook 02 — Dtype Conversion & Data Quality Audit

**Input:** accepted_columns_curtail.csv — 2,260,701 rows × 28 columns  
**Output:** accepted_dtypedchanged_coldrop.csv — 2,260,701 rows × 28 columns  
**Purpose:** Assign correct pandas dtypes, reduce memory footprint, and audit 
data quality issues for downstream sentinel treatment.

---

### Key Decisions

- `term` and `emp_length` parsed from raw text strings to numeric integers 
  (`term_months`, `emp_length_yrs`). Original text columns dropped after extraction.
- `int_rate` and `revol_util` converted from percentage format to decimal 
  (e.g. 13.99 → 0.1399) for numerical consistency.
- Nullable `Int64` used for count-based columns to preserve NaN without 
  float-casting (avoids silent loss of missingness information).
- Memory reduced from 1.89 GB → 0.51 GB (73% reduction) through correct 
  dtype assignment.
- Sentinel detection performed on `dti`, `annual_inc`, `revol_util` — 
  outliers and impossible values identified. Treatment deferred to Notebook 03.

---

### Note on Parquet Save Failure

At the end of this notebook, a parquet save was attempted to preserve dtypes 
across sessions. This failed due to a version incompatibility between 
pandas 2.x and pyarrow 22.0.0:

ArrowKeyError: No type extension with name arrow.py_extension_type found

This is a known conflict where pyarrow 22.0.0 does not correctly handle 
pandas' nullable Int64 extension types during serialization.

**Resolution:** The dataset was saved as CSV (dtypes not preserved). In 
Notebook 03, the CSV is reloaded, dtypes are re-applied using the same 
conversion logic, sentinel treatment is performed, and the final output is 
saved successfully as parquet using a compatible engine configuration.

**This notebook's output (the CSV) serves as the input to Notebook 03. 
The dtype conversion logic established here is carried forward intact.**

In [3]:
import pandas as pd
import numpy as np
import os

input_path = "/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_columns_curtail.csv"
output_path = "/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv"

print("Loading file:", input_path)
df = pd.read_csv(input_path, low_memory=False)
print(f"✅ Loaded {len(df):,} rows and {len(df.columns)} columns")


# DATE COLUMNS

for col, fmt in [("issue_d", "%b-%Y"), ("earliest_cr_line", "%b-%Y")]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format=fmt, errors="coerce")


# TERM  -> numeric months

if "term" in df.columns:
    df["term_months"] = (
        df["term"].astype(str).str.extract(r"(\d+)", expand=False)
        .astype(float)
        .astype("Int64")
    )


# EMPLOYMENT LENGTH  -> numeric years
# handles "10+ years", "< 1 year", "n/a", etc.

if "emp_length" in df.columns:
    s = df["emp_length"].astype(str).str.lower().str.strip()
    s = s.replace({"nan": None, "n/a": None})
    # convert "< 1 year" → "0", "10+ years" → "10"
    s = s.str.replace("< 1", "0", regex=False)
    s = s.str.replace("10+", "10", regex=False)
    emp_num = s.str.extract(r"(\d+)", expand=False)
    df["emp_length_yrs"] = pd.to_numeric(emp_num, errors="coerce").astype("Int64")


# FICO RANGES  -> Int64

for col in ["fico_range_low", "fico_range_high"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")


# COUNT-LIKE FLOATS  -> Int64

count_cols = ["delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "total_acc"]
for c in count_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")


# DEFAULT FLAG  -> Int64

if "default_flag" in df.columns:
    df["default_flag"] = pd.to_numeric(df["default_flag"], errors="coerce").astype("Int64")


# FLOAT (continuous) COLUMNS

float_cols = [
    "int_rate", "revol_util", "dti",
    "loan_amnt", "funded_amnt", "installment", "annual_inc"
]
for c in float_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")


# CATEGORICAL COLUMNS

cat_cols = [
    "grade", "sub_grade", "home_ownership", "verification_status",
    "purpose", "addr_state", "application_type", "loan_status"
]
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype("category")


# ID  -> string

if "id" in df.columns:
    df["id"] = df["id"].astype(str)


# RESULTS

print("\n✅ Dtype conversion complete.")
print(df.dtypes)
mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"\n💾 Memory used after dtype conversion: {mem_gb:.2f} GB")


# SAVE

os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"🎯 Saved typed DataFrame to {output_path}")


Loading file: /Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_columns_curtail.csv
✅ Loaded 2,260,701 rows and 28 columns

✅ Dtype conversion complete.
id                             object
loan_amnt                     float64
funded_amnt                   float64
term                           object
int_rate                      float64
installment                   float64
grade                        category
sub_grade                    category
emp_length                     object
home_ownership               category
annual_inc                    float64
verification_status          category
issue_d                datetime64[ns]
loan_status                  category
purpose                      category
addr_state                   category
dti                           float64
delinq_2yrs                     Int64
earliest_cr_line       datetime64[ns]
fico_range_low                  Int64
fico_range_high                 Int64
inq_last_6mths                  Int64


In [8]:
rf = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv", low_memory=False)
rf[['']].head(100)

,earliest_cr_line,issue_d
0,2003-08-01,2015-12-01
1,1999-12-01,2015-12-01
2,2000-08-01,2015-12-01
3,2008-09-01,2015-12-01
4,1998-06-01,2015-12-01
...,...,...
95,1998-01-01,2015-12-01
96,2008-05-01,2015-12-01
97,2001-03-01,2015-12-01
98,1994-04-01,2015-12-01


In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv")

# -------------------------
# Count missing values per column
# -------------------------
missing_counts = df.isna().sum().sort_values(ascending=False)
print("\n Missing values per column (sorted):")
print(missing_counts[missing_counts > 0])

# -------------------------
# Key numeric columns to check for sentinels or outliers
# -------------------------
cols_to_check = ["dti", "annual_inc", "revol_util", "loan_amnt", "int_rate"]

print("\n Checking potential sentinels / outliers in key numeric columns:\n")

for col in cols_to_check:
    if col in df.columns:
        print(f"=== {col} ===")
        # Convert to numeric (ignore errors)
        series = pd.to_numeric(df[col], errors="coerce")
        
        # Summary stats
        desc = series.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
        print(desc)
        
        # Show top 10 largest unique values (helps spot sentinels like 999)
        top_vals = series.value_counts(dropna=False).head(10)
        print("\nTop 10 values:")
        print(top_vals)
        
        # Check proportion of potential sentinel values (>= 900)
        sentinel_mask = series >= 900
        if sentinel_mask.any():
            count_sentinel = sentinel_mask.sum()
            perc_sentinel = 100 * count_sentinel / len(series)
            print(f"\n⚠️ Found {count_sentinel:,} sentinel-like values (>=900), about {perc_sentinel:.2f}% of total rows.")
        else:
            print("\n No sentinel-like values >=900 found.")
        
        print("\n" + "-"*60 + "\n")


/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_11391/2746078079.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv")



 Missing values per column (sorted):
default_flag           915351
emp_length_yrs         146940
emp_length             146940
revol_util               1835
dti                      1744
inq_last_6mths             63
total_acc                  62
pub_rec                    62
open_acc                   62
earliest_cr_line           62
delinq_2yrs                62
annual_inc                 37
term_months                33
application_type           33
fico_range_high            33
fico_range_low             33
addr_state                 33
loan_amnt                  33
purpose                    33
loan_status                33
issue_d                    33
verification_status        33
home_ownership             33
sub_grade                  33
grade                      33
installment                33
int_rate                   33
term                       33
funded_amnt                33
dtype: int64

 Checking potential sentinels / outliers in key numeric columns:

=== dti ===


In [2]:
import pandas as pd


df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv")  

# -------------------------------
# Drop redundant text columns
# -------------------------------
cols_to_drop = ["term", "emp_length"]

df = df.drop(columns=cols_to_drop, errors="ignore")
print(f" Dropped columns: {cols_to_drop}")

# -------------------------------
# Convert percentage columns
# -------------------------------
# Convert interest rate and revolving utilization from % to decimal 
for col in ["int_rate", "revol_util"]:
    if col in df.columns:
        # Convert to numeric safely (handles any stray '%' text)
        df[col] = pd.to_numeric(df[col], errors="coerce")
        # Divide by 100 to get fraction form
        df[col] = df[col] / 100

print("\n Converted int_rate and revol_util from % to decimal fractions")

# -------------------------------
# Save the intermediate cleaned dataset
# -------------------------------
output_path = "/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv"  
df.to_csv(output_path, index=False)

print(f"\n Done! Cleaned dataset saved to: {output_path}")
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")


/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_11391/653702249.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv")


 Dropped columns: ['term', 'emp_length']

 Converted int_rate and revol_util from % to decimal fractions

 Done! Cleaned dataset saved to: /Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv
Rows: 2,260,701 | Columns: 28


In [3]:
df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv")
df.info()
df.head()

/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_11391/1343741582.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 28 columns):
 #   Column               Dtype  
---  ------               -----  
 0   id                   object 
 1   loan_amnt            float64
 2   funded_amnt          float64
 3   int_rate             float64
 4   installment          float64
 5   grade                object 
 6   sub_grade            object 
 7   home_ownership       object 
 8   annual_inc           float64
 9   verification_status  object 
 10  issue_d              object 
 11  loan_status          object 
 12  purpose              object 
 13  addr_state           object 
 14  dti                  float64
 15  delinq_2yrs          float64
 16  earliest_cr_line     object 
 17  fico_range_low       float64
 18  fico_range_high      float64
 19  inq_last_6mths       float64
 20  open_acc             float64
 21  pub_rec              float64
 22  revol_util           float64
 23  total_acc            float64
 24

,id,loan_amnt,funded_amnt,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,...,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_util,total_acc,application_type,default_flag,term_months,emp_length_yrs
0,68407277,3600.0,3600.0,0.1399,123.03,C,C4,MORTGAGE,55000.0,Not Verified,...,679.0,1.0,7.0,0.0,0.297,13.0,Individual,0.0,36.0,10.0
1,68355089,24700.0,24700.0,0.1199,820.28,C,C1,MORTGAGE,65000.0,Not Verified,...,719.0,4.0,22.0,0.0,0.192,38.0,Individual,0.0,36.0,10.0
2,68341763,20000.0,20000.0,0.1078,432.66,B,B4,MORTGAGE,63000.0,Not Verified,...,699.0,0.0,6.0,0.0,0.562,18.0,Joint App,0.0,60.0,10.0
3,66310712,35000.0,35000.0,0.1485,829.90,C,C5,MORTGAGE,110000.0,Source Verified,...,789.0,0.0,13.0,0.0,0.116,17.0,Individual,NaN,60.0,10.0
4,68476807,10400.0,10400.0,0.2245,289.91,F,F1,MORTGAGE,104433.0,Source Verified,...,699.0,3.0,12.0,0.0,0.645,35.0,Individual,0.0,60.0,3.0


In [4]:
df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv")
df.info()
df.head()

/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_11391/1435506672.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 30 columns):
 #   Column               Dtype  
---  ------               -----  
 0   id                   object 
 1   loan_amnt            float64
 2   funded_amnt          float64
 3   term                 object 
 4   int_rate             float64
 5   installment          float64
 6   grade                object 
 7   sub_grade            object 
 8   emp_length           object 
 9   home_ownership       object 
 10  annual_inc           float64
 11  verification_status  object 
 12  issue_d              object 
 13  loan_status          object 
 14  purpose              object 
 15  addr_state           object 
 16  dti                  float64
 17  delinq_2yrs          float64
 18  earliest_cr_line     object 
 19  fico_range_low       float64
 20  fico_range_high      float64
 21  inq_last_6mths       float64
 22  open_acc             float64
 23  pub_rec              float64
 24

,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_util,total_acc,application_type,default_flag,term_months,emp_length_yrs
0,68407277,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,...,679.0,1.0,7.0,0.0,29.7,13.0,Individual,0.0,36.0,10.0
1,68355089,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,...,719.0,4.0,22.0,0.0,19.2,38.0,Individual,0.0,36.0,10.0
2,68341763,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,...,699.0,0.0,6.0,0.0,56.2,18.0,Joint App,0.0,60.0,10.0
3,66310712,35000.0,35000.0,60 months,14.85,829.90,C,C5,10+ years,MORTGAGE,...,789.0,0.0,13.0,0.0,11.6,17.0,Individual,NaN,60.0,10.0
4,68476807,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,...,699.0,3.0,12.0,0.0,64.5,35.0,Individual,0.0,60.0,3.0


In [5]:
pip install pyarrow

  Obtaining dependency information for pyarrow from https://files.pythonhosted.org/packages/af/63/ba23862d69652f85b615ca14ad14f3bcfc5bf1b99ef3f0cd04ff93fdad5a/pyarrow-22.0.0-cp312-cp312-macosx_12_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 3.8 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
 #dtype_convert_and_save_parquet.py
import pandas as pd
import #numpy as np
import os
import sys


INPUT_PATH = "/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv"        
OUTPUT_PARQUET = "/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldropp.parquet"     


# ---------- Load input ----------
print("Loading input file:", INPUT_PATH)
ext = os.path.splitext(INPUT_PATH)[1].lower()
if ext in [".parquet", ".pq"]:
    df = pd.read_parquet(INPUT_PATH)
else:
    # CSV or other delimited text
    df = pd.read_csv(INPUT_PATH, low_memory=False)

print(f"Loaded rows={len(df):,}, cols={len(df.columns)}")

# ---------- Desired dtype specification ----------
# Floats  want kept as float64
float_cols = [
    "loan_amnt", "funded_amnt", "int_rate", "installment",
    "annual_inc", "dti", "revol_util"
]

# Nullable integer columns (pandas Int64)
int_cols = [
    "delinq_2yrs", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "pub_rec", "total_acc", "term_months", "emp_length_yrs",
    "default_flag"
]

# Categorical columns
cat_cols = [
    "grade", "sub_grade", "home_ownership", "verification_status",
    "purpose", "addr_state", "application_type", "loan_status"
]

# Datetime columns
date_cols = ["issue_d", "earliest_cr_line"]

# ID column
id_col = "id"

# ---------- Helper: safe existence filter ----------
float_cols = [c for c in float_cols if c in df.columns]
int_cols = [c for c in int_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]
date_cols = [c for c in date_cols if c in df.columns]
id_col = id_col if id_col in df.columns else None

print("Will convert:")
print("  floats:", float_cols)
print("  ints  :", int_cols)
print("  cats  :", cat_cols)
print("  dates :", date_cols)
if id_col:
    print("  id    :", id_col)
print("")

# ---------- Convert floats (safe) ----------
for c in float_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")

# ---------- Convert nullable integers ----------
for c in int_cols:
    # coerce non-numeric to NaN then Int64
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

# ---------- Convert categories ----------
for c in cat_cols:
    # If object-like, strip whitespace then convert; preserve NaN
    try:
        df[c] = df[c].astype("category")
    except Exception:
        # fallback: convert to string then to category
        df[c] = df[c].astype(str).replace("nan", pd.NA)
        df[c] = df[c].astype("category")

# ---------- Convert datetimes (try specific format then fallback) ----------
for c in date_cols:
    # try expected format like "Dec-2015" first, then fallback
    df[c] = pd.to_datetime(df[c], format="%b-%Y", errors="coerce")
    # fallback for any remaining unparsable strings
    df.loc[df[c].isna() & df.index.notnull(), c] = pd.to_datetime(
        df.loc[df[c].isna(), c].astype(str), errors="coerce", infer_datetime_format=True
    )

# ---------- ID to string ----------
if id_col:
    df[id_col] = df[id_col].astype(str)

# ---------- Final dtype check & report ----------
print("\nAfter conversion: dtype summary (first 40 shown)")
print(df.dtypes.head(40))
mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"\nMemory used: {mem_gb:.2f} GB")

# ---------- Save to parquet (preserve dtypes) ----------
out_dir = os.path.dirname(OUTPUT_PARQUET)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)

try:
    print(f"\nSaving to parquet: {OUTPUT_PARQUET}")
    df.to_parquet(OUTPUT_PARQUET, index=False)
    print("Saved parquet successfully.")
except Exception as e:
    print("Failed to save parquet. Error:", e)
    print("\nMost common cause: missing parquet engine (pyarrow or fastparquet).")
    print("Install pyarrow with: pip install pyarrow")
    # As a fallback, save as CSV (will lose dtypes on reload)
    fallback_csv = os.path.splitext(OUTPUT_PARQUET)[0] + "_fallback.csv"
    print(f"Saving fallback CSV to: {fallback_csv} (this will NOT preserve pandas dtypes).")
    df.to_csv(fallback_csv, index=False)
    print("Fallback CSV saved. Please install pyarrow to use parquet and preserve dtypes.")
    raise

# ---------- Quick verify reload and show dtypes preserved ----------
try:
    df_check = pd.read_parquet(OUTPUT_PARQUET)
    print("\nVerified parquet reload. Dtypes preserved:")
    print(df_check.dtypes.head(40))
except Exception as e:
    print("Warning: could not reload parquet to verify. Error:", e)

print("\nDone.")


Loading input file: /Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_dtypedchanged_coldrop.csv
Loaded rows=2,260,701, cols=28
Will convert:
  floats: ['loan_amnt', 'funded_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'revol_util']
  ints  : ['delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc', 'term_months', 'emp_length_yrs', 'default_flag']
  cats  : ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'application_type', 'loan_status']
  dates : ['issue_d', 'earliest_cr_line']
  id    : id



/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_11391/182227067.py:89: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df.loc[df[c].isna() & df.index.notnull(), c] = pd.to_datetime(
/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_11391/182227067.py:89: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df.loc[df[c].isna() & df.index.notnull(), c] = pd.to_datetime(



After conversion: dtype summary (first 40 shown)
id                             object
loan_amnt                     float64
funded_amnt                   float64
int_rate                      float64
installment                   float64
grade                        category
sub_grade                    category
home_ownership               category
annual_inc                    float64
verification_status          category
issue_d                datetime64[ns]
loan_status                  category
purpose                      category
addr_state                   category
dti                           float64
delinq_2yrs                     Int64
earliest_cr_line       datetime64[ns]
fico_range_low                  Int64
fico_range_high                 Int64
inq_last_6mths                  Int64
open_acc                        Int64
pub_rec                         Int64
revol_util                    float64
total_acc                       Int64
application_type             category


ArrowKeyError: No type extension with name arrow.py_extension_type found